# CSE457 – Lab 3: Nhận dạng tiếng nói nâng cao với HMM

- **Họ tên:** Phạm Duy Long  **MSSV:** 2251061829  **Lớp:** 64CNTT1
- **Bài toán:** nhận dạng 5 lệnh tiếng Việt tách rời **mở, đóng, lên, xuống, dừng** bằng whole-word Gaussian HMM trái-sang-phải.
- **Pipeline:** `WAV → cắt silence → MFCC 13 + Δ + Δ² (39 chiều) + CMVN → 1 HMM/từ (Baum-Welch) → argmax log P(X|λ_w) → (tuỳ chọn) bigram re-ranking`

Cấu trúc notebook bám theo mục 4 của đề:
**A** dữ liệu & đặc trưng · **B** khởi tạo HMM · **C** huấn luyện · **D** nhận dạng & Viterbi · **E** so sánh DTW ·
**F** lexicon · **G** bigram & MAP re-ranking · **H** đánh giá & robustness · **Thí nghiệm E1–E6** · **Trả lời câu hỏi báo cáo**.

> Chạy: `Kernel → Restart & Run All`. Mọi bảng/hình được lưu vào `figures/`, `results/`, model vào `models/`.

In [ ]:
import time, math, pickle, warnings, logging
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import soundfile as sf
from scipy.signal import butter, sosfilt
from scipy.spatial.distance import cdist
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
import hmmlearn
from hmmlearn.hmm import GaussianHMM

warnings.filterwarnings("ignore")
logging.getLogger("hmmlearn").setLevel(logging.ERROR)
pd.set_option("display.width", 160)
np.random.seed(0)
print("numpy", np.__version__, "| librosa", librosa.__version__, "| hmmlearn", hmmlearn.__version__)

# ---------------- Tham số chung (giống nhau cho train và test) ----------------
FS = 16000
WIN = int(0.025 * FS)          # 25 ms = 400 mẫu
HOP = int(0.010 * FS)          # 10 ms = 160 mẫu
NFFT = 512
N_MELS = 26
N_MFCC = 13
PREEMPH = 0.97
TRIM_DB = 30                   # librosa.effects.trim: bỏ phần thấp hơn đỉnh 30 dB
TRIM_PAD_MS = 30               # chừa 30 ms mỗi phía để không mất phụ âm đầu/cuối

N_STATES = 5                   # baseline: 5 state/word
SELF_LOOP = 0.7                # a_ii khởi tạo
N_ITER = 80                    # số vòng Baum-Welch tối đa
TOL = 1e-3                     # dừng khi log-likelihood tăng < TOL
MIN_COVAR = 1e-3

LABELS = ["mo", "dong", "len", "xuong", "dung"]
VN = {"mo": "mở", "dong": "đóng", "len": "lên", "xuong": "xuống", "dung": "dừng"}

DATASET = Path("dataset")
FIG = Path("figures");   FIG.mkdir(exist_ok=True)
RES = Path("results");   RES.mkdir(exist_ok=True)
MODELS = Path("models"); MODELS.mkdir(exist_ok=True)

## A. Chuẩn hoá dữ liệu và trích đặc trưng

Dữ liệu: `dataset/train/<nhãn>/<nhãn>_01..06.wav` (6 file train) và `dataset/test/<nhãn>/<nhãn>_07..08.wav` (2 file test), WAV mono 16 kHz.
Train và test nằm ở **hai thư mục riêng**, notebook kiểm tra để không có file nào xuất hiện ở cả hai (không leakage).
Thư mục `dataset/mismatch/` (tuỳ chọn) dùng cho thí nghiệm E5.

In [ ]:
def list_split(split):
    return {lab: sorted((DATASET / split / lab).glob("*.wav")) for lab in LABELS}

if not (DATASET / "train").exists():
    raise FileNotFoundError("Chưa có dữ liệu. Chạy `python record_dataset.py` để thu âm vào dataset/train và dataset/test.")

train_files = list_split("train")
test_files = list_split("test")
mismatch_files = list_split("mismatch") if (DATASET / "mismatch").exists() else None

train_items = [(f, lab) for lab in LABELS for f in train_files[lab]]
test_items = [(f, lab) for lab in LABELS for f in test_files[lab]]

# Kiểm tra leakage: không trùng đường dẫn, không trùng nội dung âm thanh
assert not {f.resolve() for f, _ in train_items} & {f.resolve() for f, _ in test_items}, "File vừa ở train vừa ở test!"
_hash = lambda f: hash(sf.read(str(f))[0].tobytes())
assert not {_hash(f) for f, _ in train_items} & {_hash(f) for f, _ in test_items}, "Có file test trùng nội dung với file train!"
for lab in LABELS:
    assert len(train_files[lab]) > 0 and len(test_files[lab]) > 0, f"Thiếu dữ liệu cho lớp {lab}"

split_table = pd.DataFrame({"lệnh": [VN[l] for l in LABELS],
                            "n_train": [len(train_files[l]) for l in LABELS],
                            "n_test": [len(test_files[l]) for l in LABELS],
                            "n_mismatch": [len(mismatch_files[l]) if mismatch_files else 0 for l in LABELS]},
                           index=LABELS)
split_table

In [ ]:
# Kiểm tra chất lượng: độ dài, đỉnh, clipping, silence đầu/cuối, sample rate gốc
rows = []
for split, items in [("train", train_items), ("test", test_items)]:
    for f, lab in items:
        info = sf.info(str(f))
        raw, _ = librosa.load(str(f), sr=FS, mono=True)
        _, (s, e) = librosa.effects.trim(raw, top_db=TRIM_DB, frame_length=WIN, hop_length=HOP)
        rows.append(dict(split=split, file=f.name, label=lab, sr_goc=info.samplerate, channels=info.channels,
                         dur_s=len(raw) / FS, peak=np.max(np.abs(raw)),
                         clip_pct=100 * np.mean(np.abs(raw) >= 0.99),
                         lead_sil_s=s / FS, trail_sil_s=(len(raw) - e) / FS, speech_s=(e - s) / FS))
quality = pd.DataFrame(rows)
quality.to_csv(RES / "A_quality.csv", index=False)
print(quality[["dur_s", "peak", "clip_pct", "lead_sil_s", "trail_sil_s", "speech_s"]].describe().round(3))
print("\nSố file bị clipping:", int((quality.clip_pct > 0).sum()),
      "| Sample rate gốc:", sorted(quality.sr_goc.unique()), "| Số kênh:", sorted(quality.channels.unique()))

**Trích đặc trưng 39 chiều** (theo mục 3.2 và 5.2 của đề):
pre-emphasis 0.97 → frame 25 ms / hop 10 ms, Hamming → 26 mel filter → 13 MFCC → Δ, Δ² (`librosa.feature.delta`, hồi quy nhiều frame) → ghép $x_t=[c_t^T,\Delta c_t^T,\Delta^2 c_t^T]^T\in\mathbb R^{39}$.

**CMVN** mặc định theo từng utterance (chỉ dùng thống kê của chính utterance đó nên không thể rò rỉ thông tin test).
Phương án `cmvn="global"` dùng mean/std tính **chỉ trên tập train** rồi áp cho mọi file.

In [ ]:
def load_raw(path):
    y, _ = librosa.load(str(path), sr=FS, mono=True)
    return y / (np.max(np.abs(y)) + 1e-9)

def trim_silence(y, top_db=TRIM_DB, pad_ms=TRIM_PAD_MS):
    _, (s, e) = librosa.effects.trim(y, top_db=top_db, frame_length=WIN, hop_length=HOP)
    pad = int(pad_ms * FS / 1000)
    return y[max(0, s - pad):min(len(y), e + pad)]

def mfcc13(y):
    y = librosa.effects.preemphasis(y, coef=PREEMPH)
    return librosa.feature.mfcc(y=y, sr=FS, n_mfcc=N_MFCC, n_fft=NFFT, win_length=WIN, hop_length=HOP,
                                window="hamming", n_mels=N_MELS, fmin=20, fmax=FS / 2)   # (13, T)

def add_deltas(M, width=9):
    T = M.shape[1]
    w = min(width, T if T % 2 == 1 else T - 1)       # width phải lẻ và <= T
    w = max(3, w)
    d1 = librosa.feature.delta(M, order=1, width=w, mode="nearest")
    d2 = librosa.feature.delta(M, order=2, width=w, mode="nearest")
    return np.vstack([M, d1, d2])                     # (39, T)

GLOBAL_STATS = {}   # được điền bằng thống kê train khi cần (cmvn="global")

def extract_feat(src, feat="mfcc39", cmvn="utt", corrupt=None):
    """src: đường dẫn WAV hoặc mảng tín hiệu. Trả về ma trận T x D (D = 39 hoặc 13)."""
    y = load_raw(src) if not isinstance(src, np.ndarray) else src
    if corrupt is not None:
        y = corrupt(y)
    y = trim_silence(y)
    M = mfcc13(y)
    F = (add_deltas(M) if feat == "mfcc39" else M).T
    if cmvn == "utt":
        F = (F - F.mean(0, keepdims=True)) / (F.std(0, keepdims=True) + 1e-8)
    elif cmvn == "cmn":                               # chỉ trừ mean (giống Lab 2)
        F = F - F.mean(0, keepdims=True)
    elif cmvn == "global":
        mu, sd = GLOBAL_STATS[feat]
        F = (F - mu) / sd
    return F.astype(np.float64)

_cache = {}
def feats_of(items, **kw):
    """Trích đặc trưng có cache. items: list (path, label)."""
    out = []
    key_kw = tuple(sorted((k, v if not callable(v) else id(v)) for k, v in kw.items()))
    for f, _ in items:
        key = (str(f), key_kw)
        if key not in _cache:
            _cache[key] = extract_feat(f, **kw)
        out.append(_cache[key])
    return out

def fit_global_stats(feat="mfcc39"):
    X = np.vstack(feats_of(train_items, feat=feat, cmvn=None))
    GLOBAL_STATS[feat] = (X.mean(0, keepdims=True), X.std(0, keepdims=True) + 1e-8)

fit_global_stats("mfcc39"); fit_global_stats("mfcc13")

# Kiểm tra shape của 3 file
for f, lab in [train_items[0], train_items[len(train_items) // 2], test_items[-1]]:
    F = extract_feat(f)
    print(f"{f.name:12s} -> shape {F.shape}")
    assert F.shape[1] == 39

In [ ]:
# Heatmap MFCC / Δ / Δ² cho 2 từ khác nhau
pair = ["mo", "xuong"]
fig, axes = plt.subplots(3, 2, figsize=(13, 8), sharex="col")
for c, lab in enumerate(pair):
    F = extract_feat(train_files[lab][0])
    for r, (name, sl) in enumerate([("MFCC c1..c13", slice(0, 13)), ("Δ-MFCC", slice(13, 26)), ("Δ²-MFCC", slice(26, 39))]):
        im = axes[r, c].imshow(F[:, sl].T, origin="lower", aspect="auto", cmap="magma", vmin=-3, vmax=3)
        axes[r, c].set_ylabel(name)
        if r == 0:
            axes[r, c].set_title(f"'{VN[lab]}' ({train_files[lab][0].name}), T = {len(F)} frame")
    axes[2, c].set_xlabel("Frame (10 ms)")
fig.colorbar(im, ax=axes, shrink=0.6, label="giá trị sau CMVN")
plt.savefig(FIG / "A_mfcc_delta_heatmap.png", dpi=150, bbox_inches="tight"); plt.show()

T_table = pd.DataFrame([dict(label=lab, split=s, T=len(F))
                        for s, items in [("train", train_items), ("test", test_items)]
                        for (f, lab), F in zip(items, feats_of(items))])
print("Số frame mỗi utterance (min / mean / max) theo lớp:")
print(T_table.groupby("label").T.agg(["min", "mean", "max"]).round(1).loc[LABELS])

**Nhận xét A.** MFCC tĩnh (hàng trên) cho biết hình dạng phổ ở từng frame: đoạn nguyên âm có các dải ngang khá ổn định, còn ranh giới phụ âm–nguyên âm và phần cuối từ đổi màu đột ngột.
Δ và Δ² gần 0 ở vùng phổ ổn định và lớn ở các điểm chuyển, tức là chúng đánh dấu *chỗ phổ đang thay đổi và đổi theo hướng nào*.
"mở" là âm tiết mở có nguyên âm dài nên vùng ổn định chiếm phần lớn thời gian; "xuống" có phụ âm xát /s/ ở đầu (năng lượng tần số cao), nguyên âm đôi "uô" và âm cuối "ng", nên Δ thể hiện nhiều điểm chuyển hơn.
Số frame T khác nhau giữa các lần nói (bảng trên), nhưng mỗi frame luôn là vector 39 chiều – HMM xử lý trực tiếp chuỗi có độ dài thay đổi.

## B. Khởi tạo HMM trái-sang-phải

- $\pi=[1,0,\dots,0]$: luôn bắt đầu ở state đầu.
- $a_{ii}=0.7$, $a_{i,i+1}=0.3$, state cuối $a_{NN}=1$; mọi chuyển khác bằng 0 (không quay lại, không nhảy cóc).
- Phát xạ: 1 Gaussian **diagonal covariance**/state. Với 39 chiều, full covariance cần $39\cdot40/2=780$ tham số/state, trong khi mỗi state chỉ có khoảng vài chục tới vài trăm frame train → ma trận hiệp phương sai dễ suy biến. Diagonal chỉ cần 39 phương sai/state, ước lượng ổn định hơn, và CMVN + DCT của MFCC đã làm các chiều gần như không tương quan.
- **Khởi tạo means/covars theo phân đoạn đều (flat start):** mỗi utterance train được chia đều thành N đoạn, đoạn k dùng để ước lượng mean/var của state k. Cách này giữ đúng thứ tự trái→phải ngay từ đầu, tốt hơn k-means (`init_params="mc"`) vì k-means gán cụm không theo thời gian.
- Baum-Welch cập nhật `params="tmc"`. Một transition bằng 0 thì $\xi_t(i,j)=0$ mọi t nên vẫn bằng 0 sau khi cập nhật – topology được giữ; vẫn kiểm tra lại sau khi fit.

In [ ]:
def lr_start_trans(n, self_loop=SELF_LOOP):
    pi = np.zeros(n); pi[0] = 1.0
    A = np.zeros((n, n))
    for i in range(n - 1):
        A[i, i], A[i, i + 1] = self_loop, 1 - self_loop
    A[-1, -1] = 1.0
    return pi, A

def segmental_init(feats, n, cov):
    segs = [[] for _ in range(n)]
    for F in feats:
        for k, part in enumerate(np.array_split(F, n)):
            if len(part):
                segs[k].append(part)
    means, covars = [], []
    for k in range(n):
        X = np.vstack(segs[k])
        means.append(X.mean(0))
        if cov == "diag":
            covars.append(X.var(0) + MIN_COVAR)
        else:
            covars.append(np.cov(X.T) + 1e-2 * np.eye(X.shape[1]))
    return np.array(means), np.array(covars)

def make_lr_hmm(n_states=N_STATES, cov="diag", feats=None):
    m = GaussianHMM(n_components=n_states, covariance_type=cov, n_iter=1, tol=0.0,
                    min_covar=MIN_COVAR, init_params="", params="tmc", random_state=0)
    m.startprob_, m.transmat_ = lr_start_trans(n_states)
    if feats is not None:
        m.means_, m.covars_ = segmental_init(feats, n_states, cov)
    return m

def check_topology(A, eps=1e-10):
    """True nếu chỉ có self-loop và i->i+1 (không quay lui, không skip)."""
    n = len(A)
    bad = [(i, j, A[i, j]) for i in range(n) for j in range(n) if A[i, j] > eps and j not in (i, i + 1)]
    return len(bad) == 0, bad

pi0, A0 = lr_start_trans(N_STATES)
print("pi khởi tạo =", pi0)
print("A khởi tạo =\n", A0)

## C. Huấn luyện một HMM cho mỗi lớp

Nối các utterance train của cùng một lớp thành `X_concat` và truyền `lengths` để Baum-Welch không coi chúng là một chuỗi liền.
Để vẽ được đường log-likelihood theo từng vòng, mỗi lần gọi `fit` chỉ chạy **1 vòng EM** (`n_iter=1`, `init_params=""` nên tham số được giữ giữa các lần gọi);
`monitor_.history[-1]` là log-likelihood của tham số trước khi cập nhật. Dừng khi mức tăng < `TOL` hoặc đạt `N_ITER` vòng.

In [ ]:
def train_hmm(feats, n_states=N_STATES, cov="diag", n_iter=N_ITER, tol=TOL):
    feats = [F for F in feats if len(F) >= n_states]
    X = np.vstack(feats); lengths = [len(F) for F in feats]
    m = make_lr_hmm(n_states, cov, feats)
    hist, converged = [], False
    for _ in range(n_iter):
        m.fit(X, lengths)
        hist.append(m.monitor_.history[-1])
        if not np.isfinite(hist[-1]):
            break
        if len(hist) > 1 and abs(hist[-1] - hist[-2]) < tol:
            converged = True
            break
    m.ll_history_ = np.array(hist)
    m.converged_ = converged
    m.n_frames_ = int(sum(lengths))
    return m

def train_all(n_states=N_STATES, cov="diag", items=None, **featkw):
    items = items if items is not None else train_items
    models, t0 = {}, time.perf_counter()
    for lab in LABELS:
        feats = feats_of([it for it in items if it[1] == lab], **featkw)
        models[lab] = train_hmm(feats, n_states=n_states, cov=cov)
    return models, time.perf_counter() - t0

FEAT = dict(feat="mfcc39", cmvn="utt")
models, hmm_train_time = train_all(**FEAT)

rows = []
for lab, m in models.items():
    ok, bad = check_topology(m.transmat_)
    rows.append(dict(label=lab, frames=m.n_frames_, iters=len(m.ll_history_), converged=m.converged_,
                     loglik_cuoi=m.ll_history_[-1], loglik_per_frame=m.ll_history_[-1] / m.n_frames_,
                     topology_ok=ok))
    with open(MODELS / f"hmm_{lab}.pkl", "wb") as fh:
        pickle.dump(m, fh)
train_summary = pd.DataFrame(rows).set_index("label")
print(f"Thời gian train 5 HMM: {hmm_train_time:.2f} s")
train_summary.round(3)

In [ ]:
for lab in LABELS:
    print(f"--- {VN[lab]} ---  startprob_ = {np.round(models[lab].startprob_, 3)}")
    print(np.round(models[lab].transmat_, 3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for lab, m in models.items():
    ax.plot(np.arange(1, len(m.ll_history_) + 1), m.ll_history_ / m.n_frames_, marker=".", label=VN[lab])
ax.set_xlabel("Vòng Baum-Welch"); ax.set_ylabel("log-likelihood / frame (train)")
ax.set_title(f"Hội tụ Baum-Welch, {N_STATES} state, diag covariance"); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig(FIG / "C_convergence.png", dpi=150); plt.show()

**Nhận xét C.** Log-likelihood train tăng đơn điệu qua các vòng (tính chất của EM) và đi ngang sau vài chục vòng.
`transmat_` sau khi fit vẫn chỉ có đường chéo và đường chéo phụ trên (cột `topology_ok`), nên topology trái-sang-phải được giữ.
Giá trị $a_{ii}$ học được phản ánh thời lượng trung bình của state: kỳ vọng số frame ở state i là $1/(1-a_{ii})$, ví dụ $a_{ii}=0.9$ ứng với khoảng 10 frame = 100 ms.

## D. Nhận dạng maximum likelihood và Viterbi alignment

$\hat w=\arg\max_w \log P(X\mid\lambda_w)$. `model.score(F)` dùng thuật toán Forward (log-domain) để tính $\log P(X\mid\lambda)$.
Bảng dưới có cả score/frame và **margin** = khoảng cách top-1 – top-2 (lớn = quyết định chắc chắn).

In [ ]:
def score_all(F, models):
    return {w: m.score(F) for w, m in models.items()}

def recognize(path, models, **featkw):
    F = extract_feat(path, **{**FEAT, **featkw})
    scores = score_all(F, models)
    return max(scores, key=scores.get), scores

def evaluate(models, items, featkw=None, corrupt=None):
    featkw = featkw or FEAT
    rows = []
    for (f, lab) in items:
        F = extract_feat(f, corrupt=corrupt, **featkw) if corrupt else feats_of([(f, lab)], **featkw)[0]
        t0 = time.perf_counter()
        s = score_all(F, models)
        dt = time.perf_counter() - t0
        order = sorted(s, key=s.get, reverse=True)
        rows.append(dict(file=f.name, true=lab, pred=order[0], T=len(F),
                         **{f"ll_{w}": s[w] for w in LABELS},
                         top1=order[0], top2=order[1],
                         margin=s[order[0]] - s[order[1]],
                         margin_per_frame=(s[order[0]] - s[order[1]]) / len(F),
                         best_per_frame=s[order[0]] / len(F),
                         true_per_frame=s[lab] / len(F),
                         t_score_ms=1000 * dt))
    df = pd.DataFrame(rows)
    df["correct"] = df.true == df.pred
    return df

hmm_train_res = evaluate(models, train_items)
hmm_test = evaluate(models, test_items)
hmm_test.to_csv(RES / "D_hmm_test_scores.csv", index=False)
print(f"HMM train accuracy = {100*hmm_train_res.correct.mean():.1f}%  |  test accuracy = {100*hmm_test.correct.mean():.1f}%")

In [ ]:
# Bảng acoustic score (log-likelihood) của 3 test utterance với cả 5 model
show_files = [test_files[l][0] for l in ["mo", "len", "dung"]]
for f in show_files:
    pred, scores = recognize(f, models)
    T = len(extract_feat(f))
    tab = pd.DataFrame({"log P(X|λ)": scores, "per frame": {k: v / T for k, v in scores.items()}})
    tab = tab.sort_values("log P(X|λ)", ascending=False)
    tab["cách top-1"] = tab["log P(X|λ)"] - tab["log P(X|λ)"].iloc[0]
    print(f"\n=== {f.name}  (đúng: {VN[f.name.split('_')[0]]}, dự đoán: {VN[pred]}, T={T}) ===")
    print(tab.round(2).rename(index=VN))

In [ ]:
# Toàn bộ test set: score của mỗi utterance với tất cả model
cols = ["file", "true", "pred"] + [f"ll_{w}" for w in LABELS] + ["margin", "margin_per_frame"]
hmm_test[cols].round(1)

Model có log-likelihood lớn nhất là model "giải thích" chuỗi đặc trưng tốt nhất: tổng (trên mọi đường trạng thái) của tích xác suất chuyển và xác suất phát xạ Gaussian lớn nhất.
Vì log-likelihood là tổng trên T frame, chênh lệch tuyệt đối giữa các model thường lên tới hàng chục–hàng trăm đơn vị log; margin nhỏ (so với T) cho thấy hai từ gần nhau về mặt âm học.

In [ ]:
def runs(states):
    """Các đoạn liên tiếp cùng state: [(state, start, length)]."""
    out, start = [], 0
    for t in range(1, len(states) + 1):
        if t == len(states) or states[t] != states[start]:
            out.append((int(states[start]), start, t - start)); start = t
    return out

viterbi_rows = []
vit_examples = [test_files["mo"][0], test_files["xuong"][0], test_files["dung"][-1]]
fig, axes = plt.subplots(len(vit_examples), 1, figsize=(11, 3.2 * len(vit_examples)))
for ax, f in zip(axes, vit_examples):
    lab = f.name.split("_")[0]
    F = extract_feat(f)
    logp, states = models[lab].decode(F, algorithm="viterbi")
    logp_fw = models[lab].score(F)
    ax.imshow(F[:, :13].T, origin="lower", aspect="auto", cmap="Greys", alpha=.55,
              extent=[-0.5, len(F) - 0.5, -0.5, N_STATES - 0.5], vmin=-3, vmax=3)
    ax.step(range(len(states)), states, where="post", color="C3", lw=2, label="Viterbi state")
    ax.set_yticks(range(N_STATES)); ax.set_yticklabels([f"q{i+1}" for i in range(N_STATES)])
    ax.set_xlabel("Frame (10 ms)"); ax.set_ylabel("State")
    mono = bool(np.all(np.diff(states) >= 0))
    ax.set_title(f"Viterbi: {f.name} với model '{VN[lab]}' | log P* = {logp:.1f}, log P(X|λ) (Forward) = {logp_fw:.1f}, path không giảm: {mono}")
    ax.legend(loc="upper left")
    for st, s0, L in runs(states):
        viterbi_rows.append(dict(file=f.name, state=f"q{st+1}", start_frame=s0, frames=L, ms=10 * L))
plt.tight_layout(); plt.savefig(FIG / "D_viterbi_paths.png", dpi=150); plt.show()
viterbi_dur = pd.DataFrame(viterbi_rows)
viterbi_dur.to_csv(RES / "D_viterbi_durations.csv", index=False)
viterbi_dur.pivot(index="file", columns="state", values="ms").fillna(0).astype(int)

In [ ]:
# So sánh: cùng utterance, decode với model đúng và model sai
f = test_files["mo"][0]; F = extract_feat(f)
rows = []
for w in LABELS:
    lp, st = models[w].decode(F, algorithm="viterbi")
    rows.append(dict(model=VN[w], viterbi_logp=lp, forward_logp=models[w].score(F),
                     state_durations=[L for _, _, L in runs(st)]))
pd.DataFrame(rows).round(1)

**Phân tích Viterbi.** Đường trạng thái tốt nhất không bao giờ giảm (đúng với topology trái-sang-phải) và mỗi state chiếm một đoạn frame liên tiếp; độ dài đoạn chính là *duration* của state.
Log-prob Viterbi luôn ≤ log-likelihood Forward, vì Forward cộng xác suất của **mọi** đường còn Viterbi chỉ lấy **một** đường tốt nhất; với HMM trái-sang-phải hai giá trị thường rất gần nhau vì một đường chiếm phần lớn khối xác suất.
Khi decode bằng model sai, Viterbi vẫn "ép" được một đường nhưng log-prob thấp hơn nhiều và các đoạn state thường lệch (state bị kéo dài/co lại bất thường).
Lưu ý: `hmmlearn` không bắt buộc đường đi phải *kết thúc* ở state cuối, nên với model sai Viterbi có thể dừng ở state đầu suốt cả utterance (bảng trên, durations chỉ có một phần tử) – đó là dấu hiệu model không khớp.
Không nên coi mỗi state là một âm vị cụ thể: model chỉ được huấn luyện ở mức từ, không có giám sát âm vị; state chỉ là một đoạn phổ tương đối ổn định do EM tự tìm ra.

## E. So sánh với DTW baseline (Lab 2)

Dùng lại bộ nhận dạng của Lab 2 trên **cùng split train/test**: 13 MFCC + CMN, khoảng cách Euclid, DTW 3 bước cục bộ, chuẩn hoá theo độ dài đường đi, nearest-template.
Mọi file train (6/lớp) đều là template.

In [ ]:
DTW_FEAT = dict(feat="mfcc13", cmvn="cmn")

def dtw_distance(X, Y):
    C = cdist(X, Y)                                # khoảng cách Euclid cục bộ
    N, M = C.shape
    D = np.full((N + 1, M + 1), np.inf); D[0, 0] = 0.0
    L = np.zeros((N + 1, M + 1), dtype=int)        # độ dài path để chuẩn hoá
    for i in range(1, N + 1):
        ci = C[i - 1]
        for j in range(1, M + 1):
            a, b, c = D[i - 1, j - 1], D[i - 1, j], D[i, j - 1]
            if a <= b and a <= c:   D[i, j], L[i, j] = ci[j - 1] + a, L[i - 1, j - 1] + 1
            elif b <= c:            D[i, j], L[i, j] = ci[j - 1] + b, L[i - 1, j] + 1
            else:                   D[i, j], L[i, j] = ci[j - 1] + c, L[i, j - 1] + 1
    return D[N, M] / L[N, M]

def dtw_build(items):
    t0 = time.perf_counter()
    templates = [(lab, F) for (f, lab), F in zip(items, feats_of(items, **DTW_FEAT))]
    return templates, time.perf_counter() - t0

def dtw_evaluate(templates, items, corrupt=None):
    rows = []
    for f, lab in items:
        X = extract_feat(f, corrupt=corrupt, **DTW_FEAT) if corrupt else feats_of([(f, lab)], **DTW_FEAT)[0]
        t0 = time.perf_counter()
        best = {w: np.inf for w in LABELS}
        for w, Y in templates:
            best[w] = min(best[w], dtw_distance(X, Y))
        dt = time.perf_counter() - t0
        order = sorted(best, key=best.get)
        rows.append(dict(file=f.name, true=lab, pred=order[0], **{f"dtw_{w}": best[w] for w in LABELS},
                         margin=best[order[1]] - best[order[0]], t_score_ms=1000 * dt))
    df = pd.DataFrame(rows); df["correct"] = df.true == df.pred
    return df

dtw_templates, dtw_train_time = dtw_build(train_items)
dtw_test = dtw_evaluate(dtw_templates, test_items)
dtw_test.to_csv(RES / "E_dtw_test_scores.csv", index=False)
print(f"DTW test accuracy = {100*dtw_test.correct.mean():.1f}%  |  HMM test accuracy = {100*hmm_test.correct.mean():.1f}%")

In [ ]:
def plot_cm(df, title, fname, ax=None):
    cm = confusion_matrix(df.true, df.pred, labels=LABELS)
    disp = ConfusionMatrixDisplay(cm, display_labels=[VN[l] for l in LABELS])
    disp.plot(cmap="Blues", ax=ax, colorbar=False)
    (ax or plt.gca()).set_title(f"{title}\naccuracy = {100*np.trace(cm)/cm.sum():.1f}%")
    return cm

def top_confusion(df):
    wrong = df[~df.correct]
    if wrong.empty:
        return "không có lỗi"
    (t, p), n = Counter(zip(wrong.true, wrong.pred)).most_common(1)[0]
    return f"{VN[t]} → {VN[p]} ({n} lần)"

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
cm_dtw = plot_cm(dtw_test, "DTW (Lab 2, 13 MFCC + CMN)", None, axes[0])
cm_hmm = plot_cm(hmm_test, f"HMM ({N_STATES} state, 39-D, diag)", None, axes[1])
plt.tight_layout(); plt.savefig(FIG / "E_confusion_dtw_vs_hmm.png", dpi=150); plt.show()

In [ ]:
n_train_per_class = int(np.mean([len(train_files[l]) for l in LABELS]))
compare = pd.DataFrame([
    dict(**{"Chỉ số": "Accuracy (test)"}, DTW=f"{100*dtw_test.correct.mean():.1f}%", HMM=f"{100*hmm_test.correct.mean():.1f}%"),
    dict(**{"Chỉ số": "Số mẫu train/lớp"}, DTW=f"{n_train_per_class} template", HMM=f"{n_train_per_class} utterance (ước lượng tham số)"),
    dict(**{"Chỉ số": "Thời gian train"}, DTW=f"{dtw_train_time:.2f} s (chỉ trích & lưu template)", HMM=f"{hmm_train_time:.2f} s (Baum-Welch 5 model)"),
    dict(**{"Chỉ số": "Thời gian test/trial"}, DTW=f"{dtw_test.t_score_ms.mean():.1f} ms", HMM=f"{hmm_test.t_score_ms.mean():.2f} ms"),
    dict(**{"Chỉ số": "Top confusion pair"}, DTW=top_confusion(dtw_test), HMM=top_confusion(hmm_test)),
    dict(**{"Chỉ số": "Giải thích alignment"}, DTW="warping path", HMM="Viterbi state path"),
    dict(**{"Chỉ số": "Bộ nhớ lúc test"}, DTW=f"{len(dtw_templates)} template", HMM=f"5 model × {N_STATES} state × (39 mean + 39 var)"),
]).set_index("Chỉ số")
compare.to_csv(RES / "E_dtw_vs_hmm.csv")
compare

In [ ]:
# Những utterance mà một phương pháp đúng còn phương pháp kia sai
diff = hmm_test[["file", "true", "pred", "margin_per_frame"]].rename(columns={"pred": "pred_HMM", "margin_per_frame": "HMM_margin/frame"})
diff["pred_DTW"] = dtw_test.pred.values
diff["DTW_margin"] = dtw_test.margin.values
diff = diff[(diff.pred_HMM == diff.true) != (diff.pred_DTW == diff.true)]
print("Số utterance hai phương pháp cho kết quả khác nhau:", len(diff))
diff.round(3)

**Nhận xét E.** DTW so sánh trực tiếp với từng template nên chỉ cần rất ít mẫu, nhưng chi phí test tỉ lệ với số template × $T_x\cdot T_y$ và quyết định phụ thuộc vào template "gần nhất" – một template xấu (nói lệch, nhiễu) có thể kéo sai cả lớp.
HMM gộp mọi utterance train thành phân bố Gaussian theo state nên mô hình hoá được độ biến thiên (phương sai) giữa các lần nói; score chỉ tốn $O(N^2T)$ cho mỗi model và không phụ thuộc số mẫu train.
Đổi lại HMM cần đủ dữ liệu để ước lượng mean/var của từng state (xem thí nghiệm E4 bên dưới: khi chỉ có 1–2 mẫu/lớp, DTW thường ổn định hơn).
Bảng "khác nhau" ở trên là các ca cụ thể để phân tích: nếu HMM đúng mà DTW sai, thường là do template gần nhất của lớp sai tình cờ khớp tốt hơn; nếu DTW đúng mà HMM sai, thường là do lớp đó có ít frame cho một số state nên Gaussian bị hẹp.

## F. Lexicon mini

File `lexicon.txt` theo định dạng `WORD PHONE_1 ... PHONE_K`. Bộ phone ở đây là tạm (âm đầu + vần + thanh điệu); khi giảng viên cung cấp inventory chuẩn chỉ cần thay cột phone.

In [ ]:
def load_lexicon(path="lexicon.txt"):
    lex = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.split("#", 1)[0].strip()
        if not line:
            continue
        word, *phones = line.split()
        assert phones, f"Từ '{word}' không có phone"
        lex.setdefault(word, []).append(phones)        # cho phép nhiều cách phát âm
    return lex

lexicon = load_lexicon()
for w, prons in lexicon.items():
    print(f"{w:6s} ({VN.get(w, w)}): {[' '.join(p) for p in prons]}")
phone_set = sorted({p for prons in lexicon.values() for pr in prons for p in pr})
print("\nPhone inventory dùng trong lexicon:", phone_set, f"({len(phone_set)} phone)")

In [ ]:
def word_hmm_from_phones(phones, states_per_phone=3, self_loop=SELF_LOOP):
    """Ghép các phone HMM trái-sang-phải thành một word HMM (chỉ minh hoạ topology).
    State cuối của phone k nối sang state đầu của phone k+1."""
    n = states_per_phone * len(phones)
    pi, A = lr_start_trans(n, self_loop)
    names = [f"{p}_{s+1}" for p in phones for s in range(states_per_phone)]
    return pi, A, names

pi_w, A_w, names = word_hmm_from_phones(lexicon["xuong"][0])
fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(A_w, cmap="Blues")
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=90)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
for k in range(1, len(lexicon["xuong"][0])):
    ax.axhline(3 * k - 0.5, color="C3", lw=.8); ax.axvline(3 * k - 0.5, color="C3", lw=.8)
ax.set_title("Word HMM 'xuống' = nối các phone HMM 3 state theo lexicon\n(đường đỏ: ranh giới phone)")
plt.tight_layout(); plt.savefig(FIG / "F_word_hmm_from_lexicon.png", dpi=150); plt.show()
print("Số state của word HMM 'xuống':", len(names))

**Giải thích F.** Với whole-word HMM, mỗi từ có model riêng nên không cần lexicon. Khi chuyển sang phone HMM, ta chỉ huấn luyện một bộ model cho các phone (dùng chung giữa mọi từ),
rồi tạo word HMM bằng cách **nối** phone HMM theo chuỗi phone trong lexicon: transition ra khỏi state cuối của phone k chính là transition vào state đầu của phone k+1 (ma trận trên có dạng khối, các khối nối nhau ở đường chéo phụ).
Nhờ đó có thể nhận dạng cả từ chưa từng có trong dữ liệu train, miễn là có trong lexicon. Lexicon trở thành **bắt buộc** khi từ vựng lớn hoặc mở (không thể thu đủ mẫu cho mọi từ), khi dùng subword/phone model và khi ghép với decoder liên tục.

## G. Bigram và MAP re-ranking

$P(w_j\mid w_i) = \dfrac{C(w_i,w_j)+k}{C(w_i)+kV}$ (add-k, k = 0.5), $V$ = kích thước từ vựng (5 lệnh + `</s>`).
Score kết hợp: $\text{Score}(w)=\frac{1}{T}\log P(X\mid\lambda_w)+\lambda_{LM}\log P(w\mid w_{prev})$ (giống mục 5.9 của đề: acoustic score chia T để hai thang điểm cùng cỡ).

In [ ]:
def load_corpus(path="corpus.txt"):
    sents = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.split("#", 1)[0].strip()
        if line:
            sents.append(["<s>"] + line.split() + ["</s>"])
    return sents

corpus = load_corpus()
counts1, counts2 = Counter(), Counter()
for s in corpus:
    counts1.update(s[:-1])                      # C(w_i) dùng làm mẫu số (từ đứng trước)
    counts2.update(zip(s[:-1], s[1:]))
unigram = Counter(w for s in corpus for w in s[1:-1])
V = len(LABELS) + 1
K_ADD = 0.5

def bigram_prob(prev, cur, k=K_ADD):
    return (counts2.get((prev, cur), 0) + k) / (counts1.get(prev, 0) + k * V)

print("Số câu:", len(corpus), "| unigram:", {VN[w]: unigram[w] for w in LABELS})
bigram_table = pd.DataFrame({cur: {prev: bigram_prob(prev, cur) for prev in ["<s>"] + LABELS} for cur in LABELS + ["</s>"]})
bigram_table.index = [VN.get(i, i) for i in bigram_table.index]
bigram_table.columns = [VN.get(c, c) for c in bigram_table.columns]
print("P(cột | hàng), add-k k=0.5 (mỗi hàng cộng lại = 1):")
bigram_table.round(3)

In [ ]:
def rerank_scores(F, prev, models, lm_weight):
    T = len(F)
    return {w: m.score(F) / T + lm_weight * math.log(bigram_prob(prev, w)) for w, m in models.items()}

# Chọn utterance có top-1 và top-2 acoustic gần nhau nhất (margin/frame nhỏ nhất)
close = hmm_test.sort_values("margin_per_frame").iloc[0]
f_close = next(f for f, _ in test_items if f.name == close.file)
F_close = extract_feat(f_close)
c1, c2 = close.top1, close.top2
print(f"Utterance: {close.file} (đúng: {VN[close.true]}) | acoustic top-1 = {VN[c1]}, top-2 = {VN[c2]}, "
      f"margin/frame = {close.margin_per_frame:.3f}")

LM_WEIGHTS = [0.0, 0.1, 0.3, 1.0, 3.0, 10.0]
rows = []
for prev in ["<s>"] + LABELS:
    row = {"từ trước": VN.get(prev, prev), f"P({VN[c1]}|prev)": bigram_prob(prev, c1), f"P({VN[c2]}|prev)": bigram_prob(prev, c2)}
    for lw in LM_WEIGHTS:
        s = rerank_scores(F_close, prev, models, lw)
        row[f"λ={lw}"] = VN[max(s, key=s.get)]
    rows.append(row)
rerank_demo = pd.DataFrame(rows).set_index("từ trước")
rerank_demo.to_csv(RES / "G_rerank_demo.csv")
rerank_demo.round(3)

In [ ]:
# Đánh giá có hệ thống: mỗi test utterance được ghép với một từ đứng trước
#  - "ngữ cảnh hợp lý": prev làm P(từ đúng | prev) lớn nhất
#  - "ngữ cảnh gây nhiễu": prev làm một từ SAI có xác suất cao nhất so với từ đúng
def helpful_prev(w):
    return max(["<s>"] + LABELS, key=lambda p: bigram_prob(p, w))

def misleading_prev(w):
    return max(["<s>"] + LABELS,
               key=lambda p: max(bigram_prob(p, o) for o in LABELS if o != w) - bigram_prob(p, w))

test_feats = feats_of(test_items, **FEAT)
LM_SWEEP = [0.0, 0.05, 0.1, 0.3, 1.0, 3.0, 10.0]
rows = []
for lw in LM_SWEEP:
    for ctx_name, ctx_fn in [("hợp lý", helpful_prev), ("gây nhiễu", misleading_prev)]:
        correct = 0
        for (f, lab), F in zip(test_items, test_feats):
            s = rerank_scores(F, ctx_fn(lab), models, lw)
            correct += max(s, key=s.get) == lab
        rows.append(dict(lm_weight=lw, context=ctx_name, accuracy=100 * correct / len(test_items)))
lm_sweep = pd.DataFrame(rows).pivot(index="lm_weight", columns="context", values="accuracy")
lm_sweep.to_csv(RES / "G_lm_weight_sweep.csv")

ax = lm_sweep.plot(marker="o", figsize=(7, 4), logx=False)
ax.set_xlabel("λ_LM"); ax.set_ylabel("Accuracy test (%)"); ax.set_title("Ảnh hưởng của λ_LM tới quyết định top-1")
ax.grid(alpha=.3); plt.tight_layout(); plt.savefig(FIG / "G_lm_weight.png", dpi=150); plt.show()
lm_sweep.round(1)

**Nhận xét G.** Acoustic score/frame của các lớp thường chỉ cách nhau vài phần mười đơn vị log, trong khi $\log P(w\mid w_{prev})$ chênh nhau cỡ 1–3 đơn vị.
- $\lambda_{LM}=0$: quyết định thuần âm học (maximum likelihood).
- $\lambda_{LM}$ nhỏ (0.05–0.3): LM chỉ "phân xử" khi hai ứng viên âm học gần nhau → có thể sửa lỗi nếu ngữ cảnh hợp lý (ví dụ sau "lên" gần như luôn là "dừng").
- $\lambda_{LM}$ lớn (1–3): prior lấn át bằng chứng âm học; với ngữ cảnh gây nhiễu, hệ thống chọn từ mà LM thích dù người nói nói từ khác → accuracy giảm mạnh, có thể sai cả những quyết định âm học vốn đúng.

Đây chỉ là minh hoạ nguyên lý Bayes $P(W\mid X)\propto P(X\mid W)P(W)$ trên từ đơn khi đã biết từ trước. Decoder tiếng nói liên tục phải tìm **đồng thời** ranh giới từ và chuỗi từ trên một mạng HMM lớn (Viterbi/beam search), không phải re-rank từng từ đã cắt sẵn.

## H. Đánh giá và thí nghiệm robustness

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
plot_cm(hmm_train_res, "HMM – tập TRAIN", None, axes[0])
plot_cm(hmm_test, "HMM – tập TEST sạch", None, axes[1])
plt.tight_layout(); plt.savefig(FIG / "H_confusion_hmm.png", dpi=150); plt.show()

per_class = hmm_test.groupby("true").correct.mean().mul(100).reindex(LABELS).rename(index=VN)
print("Accuracy theo lớp (test, %):"); print(per_class.round(1).to_string())
print(f"\nTrain acc = {100*hmm_train_res.correct.mean():.1f}%, test acc = {100*hmm_test.correct.mean():.1f}%  "
      f"→ chênh lệch {100*(hmm_train_res.correct.mean()-hmm_test.correct.mean()):.1f} điểm")

**Điều kiện mismatch** (đề yêu cầu ít nhất một): nhiễu trắng cộng vào tín hiệu test với SNR 20/10/5 dB, và mô phỏng kênh "điện thoại/micro khác" bằng lọc thông dải 300–3400 Hz + giảm biên độ.
Nếu có thư mục `dataset/mismatch/` (người nói khác/đứng xa micro), tập này cũng được đánh giá. Model **không** được huấn luyện lại – chỉ thay đổi dữ liệu test.

In [ ]:
def add_noise(snr_db, seed=0):
    def f(y):
        rng = np.random.default_rng(seed)
        p_sig = np.mean(y ** 2)
        noise = rng.standard_normal(len(y)) * np.sqrt(p_sig / 10 ** (snr_db / 10))
        return y + noise
    f.__name__ = f"noise_{snr_db}dB"
    return f

_sos_tel = butter(6, [300, 3400], btype="bandpass", fs=FS, output="sos")
def telephone(y):
    return 0.3 * sosfilt(_sos_tel, y)

conditions = [("Sạch", None), ("Nhiễu trắng 20 dB", add_noise(20)), ("Nhiễu trắng 10 dB", add_noise(10)),
              ("Nhiễu trắng 5 dB", add_noise(5)), ("Kênh 300–3400 Hz", telephone)]
mismatch_items = [(f, lab) for lab in LABELS for f in mismatch_files[lab]] if mismatch_files else []

rows, mismatch_results = [], {}
for name, cor in conditions:
    h = hmm_test if cor is None else evaluate(models, test_items, corrupt=cor)
    d = dtw_test if cor is None else dtw_evaluate(dtw_templates, test_items, corrupt=cor)
    mismatch_results[name] = h
    rows.append(dict(condition=name, HMM=100 * h.correct.mean(), DTW=100 * d.correct.mean()))
if mismatch_items:
    h = evaluate(models, mismatch_items); d = dtw_evaluate(dtw_templates, mismatch_items)
    mismatch_results["Tập mismatch thu thật"] = h
    rows.append(dict(condition="Tập mismatch thu thật", HMM=100 * h.correct.mean(), DTW=100 * d.correct.mean()))
robust = pd.DataFrame(rows).set_index("condition")
robust["HMM giảm (điểm)"] = robust.HMM.iloc[0] - robust.HMM
robust["DTW giảm (điểm)"] = robust.DTW.iloc[0] - robust.DTW
robust.to_csv(RES / "H_robustness.csv")
robust.round(1)

In [ ]:
# CMVN theo utterance vs global-train CMVN dưới mismatch kênh (bài mở rộng)
models_g, _ = train_all(feat="mfcc39", cmvn="global")
rows = []
for name, cor in conditions:
    acc_u = 100 * mismatch_results[name].correct.mean()
    acc_g = 100 * evaluate(models_g, test_items, featkw=dict(feat="mfcc39", cmvn="global"), corrupt=cor).correct.mean()
    rows.append(dict(condition=name, utt_CMVN=acc_u, global_train_CMVN=acc_g))
cmvn_cmp = pd.DataFrame(rows).set_index("condition")
cmvn_cmp.to_csv(RES / "H_cmvn_compare.csv")
cmvn_cmp.round(1)

**Phân bố score đúng lớp vs sai lớp và tiêu chí reject.**
Vì argmax luôn chọn một lớp, hệ thống cần một tiêu chí để trả lời "unknown". Ta mô phỏng từ ngoài từ vựng (OOV) bằng **leave-one-class-out**: bỏ model của lớp c, khi đó các test utterance của lớp c là OOV với 4 model còn lại.
Hai đại lượng được dùng: (1) log-likelihood/frame của top-1, (2) margin/frame top-1 – top-2.

In [ ]:
pool = pd.concat([mismatch_results[n] for n, _ in conditions[:3]], ignore_index=True)   # sạch + SNR 20/10 dB
in_vocab = pd.DataFrame(dict(best=pool.best_per_frame, margin=pool.margin_per_frame, correct=pool.correct))

rows = []
for c in LABELS:
    sub = {w: m for w, m in models.items() if w != c}
    for (f, lab), F in zip(test_items, test_feats):
        if lab != c:
            continue
        s = sorted((m.score(F) / len(F) for m in sub.values()), reverse=True)
        rows.append(dict(best=s[0], margin=s[0] - s[1]))
oov = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bins = 25
axes[0].hist(in_vocab.best[in_vocab.correct], bins=bins, alpha=.6, label="trong từ vựng – nhận đúng")
axes[0].hist(in_vocab.best[~in_vocab.correct], bins=bins, alpha=.6, label="trong từ vựng – nhận sai")
axes[0].hist(oov.best, bins=bins, alpha=.6, label="OOV (leave-one-class-out)")
axes[0].set_xlabel("log-likelihood/frame của top-1"); axes[0].legend(); axes[0].grid(alpha=.3)
axes[1].hist(in_vocab.margin[in_vocab.correct], bins=bins, alpha=.6, label="nhận đúng")
axes[1].hist(in_vocab.margin[~in_vocab.correct], bins=bins, alpha=.6, label="nhận sai")
axes[1].hist(oov.margin, bins=bins, alpha=.6, label="OOV")
axes[1].set_xlabel("margin/frame (top-1 – top-2)"); axes[1].legend(); axes[1].grid(alpha=.3)
plt.suptitle("Phân bố score: đúng lớp, sai lớp và OOV (test sạch + nhiễu 20/10 dB)")
plt.tight_layout(); plt.savefig(FIG / "H_score_distribution.png", dpi=150); plt.show()

In [ ]:
# Quét ngưỡng reject trên log-likelihood/frame: tỉ lệ từ chối nhầm (FRR) vs chấp nhận nhầm OOV (FAR)
ths = np.linspace(min(in_vocab.best.min(), oov.best.min()), max(in_vocab.best.max(), oov.best.max()), 200)
frr = np.array([(in_vocab.best[in_vocab.correct] < t).mean() for t in ths])
far = np.array([(oov.best >= t).mean() for t in ths])
i_eer = int(np.argmin(np.abs(frr - far)))
theta = ths[i_eer]

plt.figure(figsize=(7, 4))
plt.plot(ths, 100 * frr, label="FRR: từ chối nhầm từ đúng"); plt.plot(ths, 100 * far, label="FAR: chấp nhận nhầm OOV")
plt.axvline(theta, color="k", ls="--", label=f"θ ≈ {theta:.2f} (EER ≈ {100*(frr[i_eer]+far[i_eer])/2:.0f}%)")
plt.xlabel("ngưỡng θ trên log-likelihood/frame"); plt.ylabel("%"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG / "H_reject_threshold.png", dpi=150); plt.show()

def recognize_with_reject(F, models, theta=theta, delta=None):
    s = {w: m.score(F) / len(F) for w, m in models.items()}
    order = sorted(s, key=s.get, reverse=True)
    if s[order[0]] < theta or (delta is not None and s[order[0]] - s[order[1]] < delta):
        return "unknown", s
    return order[0], s

print(f"Ngưỡng đề xuất θ = {theta:.3f} (log-likelihood/frame). Tại ngưỡng này: "
      f"FRR = {100*frr[i_eer]:.1f}%, FAR(OOV) = {100*far[i_eer]:.1f}%")

**Nhận xét H.** So sánh train/test accuracy cho biết mức overfit; bảng robustness cho thấy nguồn mismatch nào làm giảm accuracy mạnh nhất.
Nhiễu trắng làm "đầy" các vùng phổ năng lượng thấp (khoảng lặng, phụ âm xát), làm lệch log mel-energy ở mọi frame nên ảnh hưởng mạnh khi SNR thấp; lọc kênh chủ yếu là một bộ lọc tuyến tính, tương đương *cộng một hằng số* vào cepstrum, nên CMVN theo utterance khử được phần lớn (so sánh bảng utt-CMVN vs global-CMVN).
**Đề xuất reject:** trả về "unknown" nếu log-likelihood/frame của top-1 < θ (ngưỡng chọn tại điểm EER ở trên), có thể kết hợp thêm margin/frame < δ để bắt các ca hai lớp sát nhau.
Dùng score **chia cho T** vì ngưỡng phải so được giữa các utterance có độ dài khác nhau. Cách bền hơn là thêm một "filler/garbage model" huấn luyện trên tiếng nói chung và so sánh tỉ số likelihood.

## Thí nghiệm bắt buộc E1–E6
Mỗi thí nghiệm chỉ thay đúng **một** yếu tố, các tham số khác giữ như baseline (5 state, 39-D, diag, utt-CMVN, 6 train/lớp).

In [ ]:
# E1: số state
rows = []
for n in [3, 5, 7]:
    ms, tt = train_all(n_states=n, **FEAT)
    rows.append(dict(n_states=n,
                     train_acc=100 * evaluate(ms, train_items).correct.mean(),
                     test_acc=100 * evaluate(ms, test_items).correct.mean(),
                     converged=f"{sum(m.converged_ for m in ms.values())}/5",
                     mean_iters=np.mean([len(m.ll_history_) for m in ms.values()]),
                     train_ll_per_frame=np.mean([m.ll_history_[-1] / m.n_frames_ for m in ms.values()]),
                     frames_per_state=np.mean([m.n_frames_ for m in ms.values()]) / n,
                     train_time_s=tt))
E1 = pd.DataFrame(rows).set_index("n_states")
E1.round(2)

In [ ]:
# E2: 13 MFCC tĩnh vs 39 chiều MFCC+Δ+Δ²
models13, _ = train_all(feat="mfcc13", cmvn="utt")
hmm13_test = evaluate(models13, test_items, featkw=dict(feat="mfcc13", cmvn="utt"))
E2 = pd.DataFrame([
    dict(feature="13 MFCC", test_acc=100 * hmm13_test.correct.mean(), top_confusion=top_confusion(hmm13_test),
         **{VN[l]: 100 * hmm13_test[hmm13_test.true == l].correct.mean() for l in LABELS}),
    dict(feature="MFCC+Δ+Δ² (39)", test_acc=100 * hmm_test.correct.mean(), top_confusion=top_confusion(hmm_test),
         **{VN[l]: 100 * hmm_test[hmm_test.true == l].correct.mean() for l in LABELS}),
]).set_index("feature")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
plot_cm(hmm13_test, "HMM 13 MFCC", None, axes[0]); plot_cm(hmm_test, "HMM 39-D", None, axes[1])
plt.tight_layout(); plt.savefig(FIG / "E2_confusion_13_vs_39.png", dpi=150); plt.show()
E2.round(1)

In [ ]:
# E3: diagonal vs full covariance (39-D)
rows = []
for cov in ["diag", "full"]:
    try:
        ms, tt = train_all(cov=cov, **FEAT)
        tr, te = evaluate(ms, train_items), evaluate(ms, test_items)
        n_par = sum(m.means_.size + (m.covars_.shape[-1] * (m.covars_.shape[-1] + 1) // 2 * m.n_components
                                     if cov == "full" else m.means_.size) for m in ms.values())
        rows.append(dict(covariance=cov, n_params_gauss=n_par, train_acc=100 * tr.correct.mean(), test_acc=100 * te.correct.mean(),
                         train_ll_per_frame=tr.true_per_frame.mean(), test_ll_per_frame=te.true_per_frame.mean(),
                         gap_train_test=tr.true_per_frame.mean() - te.true_per_frame.mean(), note="ok"))
    except Exception as ex:
        rows.append(dict(covariance=cov, note=f"lỗi: {type(ex).__name__}: {ex}"[:120]))
E3 = pd.DataFrame(rows).set_index("covariance")
E3.round(2)

In [ ]:
# E4: DTW vs HMM khi thay đổi số mẫu train/lớp
rows = []
max_k = min(len(train_files[l]) for l in LABELS)
for k in sorted({1, 2, 3, 4, max_k}):
    if k > max_k:
        continue
    sub = [(f, lab) for lab in LABELS for f in train_files[lab][:k]]
    ms, tt = train_all(items=sub, **FEAT)
    h = evaluate(ms, test_items)
    tmpl, dt_build = dtw_build(sub)
    d = dtw_evaluate(tmpl, test_items)
    rows.append(dict(n_train_per_class=k, HMM_acc=100 * h.correct.mean(), DTW_acc=100 * d.correct.mean(),
                     HMM_train_s=tt, HMM_test_ms=h.t_score_ms.mean(), DTW_test_ms=d.t_score_ms.mean()))
E4 = pd.DataFrame(rows).set_index("n_train_per_class")
ax = E4[["HMM_acc", "DTW_acc"]].plot(marker="o", figsize=(7, 4))
ax.set_ylabel("Accuracy test (%)"); ax.set_xlabel("Số utterance train mỗi lớp"); ax.grid(alpha=.3)
ax.set_title("E4: DTW vs HMM theo lượng dữ liệu train")
plt.tight_layout(); plt.savefig(FIG / "E4_dtw_vs_hmm_data_size.png", dpi=150); plt.show()
E4.round(2)

In [ ]:
# E5: mismatch (đã chạy ở mục H) và E6: λ_LM (đã chạy ở mục G)
E5 = robust
E6 = lm_sweep
exp_summary = pd.DataFrame([
    dict(exp="E1", bien="n_states = 3/5/7",
         ket_qua=", ".join(f"{n}: {r.test_acc:.0f}% (hội tụ {r.converged})" for n, r in E1.iterrows())),
    dict(exp="E2", bien="13 MFCC vs 39-D",
         ket_qua=f"13: {E2.test_acc.iloc[0]:.0f}% | 39: {E2.test_acc.iloc[1]:.0f}%"),
    dict(exp="E3", bien="diag vs full",
         ket_qua=" | ".join(f"{c}: train {r.get('train_acc', float('nan')):.0f}% / test {r.get('test_acc', float('nan')):.0f}%"
                            for c, r in E3.iterrows())),
    dict(exp="E4", bien="DTW vs HMM (số mẫu train)",
         ket_qua=" | ".join(f"k={k}: HMM {r.HMM_acc:.0f}% / DTW {r.DTW_acc:.0f}%" for k, r in E4.iterrows())),
    dict(exp="E5", bien="mismatch",
         ket_qua=" | ".join(f"{c}: HMM {r.HMM:.0f}%" for c, r in E5.iterrows())),
    dict(exp="E6", bien="λ_LM",
         ket_qua=" | ".join(f"λ={lw}: {r['hợp lý']:.0f}%/{r['gây nhiễu']:.0f}%" for lw, r in E6.iterrows())
         + " (ngữ cảnh hợp lý / gây nhiễu)"),
]).set_index("exp")
exp_summary.to_csv(RES / "experiments_summary.csv")
for name, df in [("E1", E1), ("E2", E2), ("E3", E3), ("E4", E4), ("E5", E5), ("E6", E6)]:
    df.to_csv(RES / f"{name}.csv")
pd.set_option("display.max_colwidth", 200)
exp_summary

In [ ]:
# Tóm tắt số liệu chính để viết báo cáo
best_n = E1.test_acc.idxmax()
worst_cond = robust.HMM.idxmin()
print(f"""
TÓM TẮT KẾT QUẢ
- Dữ liệu: {len(train_items)} file train, {len(test_items)} file test, {len(LABELS)} lớp.
- HMM baseline ({N_STATES} state, 39-D, diag): train {100*hmm_train_res.correct.mean():.1f}% / test {100*hmm_test.correct.mean():.1f}%; lỗi phổ biến nhất: {top_confusion(hmm_test)}.
- DTW (Lab 2) cùng split: test {100*dtw_test.correct.mean():.1f}%; lỗi phổ biến nhất: {top_confusion(dtw_test)}.
- E1: số state tốt nhất trên test = {best_n} ({E1.test_acc.max():.1f}%).
- E2: 13 MFCC {E2.test_acc.iloc[0]:.1f}% vs 39-D {E2.test_acc.iloc[1]:.1f}%.
- E5: điều kiện làm giảm mạnh nhất = {worst_cond} (HMM còn {robust.HMM.min():.1f}%).
- Reject: θ = {theta:.2f} log-lik/frame, FRR ≈ FAR ≈ {100*(frr[i_eer]+far[i_eer])/2:.0f}%.
""")

## Trả lời câu hỏi báo cáo

**1. Forward và Viterbi khác nhau ở phép toán nào, trả lời câu hỏi gì?**
Cùng quy hoạch động trên trellis, nhưng Forward dùng **tổng** $\alpha_t(j)=b_j(x_t)\sum_i\alpha_{t-1}(i)a_{ij}$ còn Viterbi dùng **max** $\delta_t(j)=b_j(x_t)\max_i\delta_{t-1}(i)a_{ij}$ và lưu back-pointer.
Forward trả lời "mô hình sinh ra chuỗi X với xác suất bao nhiêu" ($P(X\mid\lambda)$, dùng để chấm điểm/nhận dạng); Viterbi trả lời "chuỗi trạng thái đơn lẻ nào khả dĩ nhất" ($Q^*$ và $P(X,Q^*\mid\lambda)$, dùng để alignment/giải mã). Luôn có $P(X,Q^*\mid\lambda)\le P(X\mid\lambda)$.

**2. $\gamma_t(i)$ và $\xi_t(i,j)$ theo ngôn ngữ "trách nhiệm".**
$\gamma_t(i)$ là phần trách nhiệm (xác suất hậu nghiệm) mà state i nhận cho việc sinh frame $x_t$, đã xét toàn bộ chuỗi quan sát; tổng theo i bằng 1. $\xi_t(i,j)$ là trách nhiệm của *chuyển* $i\to j$ giữa t và t+1.
Trong M-step mỗi frame đóng góp vào mean/var của state i theo trọng số $\gamma_t(i)$ (gán "mềm" thay vì gán cứng như Viterbi training), còn $a_{ij}$ mới = tổng trách nhiệm của chuyển $i\to j$ / tổng trách nhiệm của state i.

**3. Vì sao Baum-Welch có thể hội tụ đến nghiệm xấu?** EM chỉ đảm bảo likelihood không giảm và hội tụ tới cực đại **cục bộ**. Nguyên nhân thực nghiệm: (a) khởi tạo kém (k-means không theo thứ tự thời gian, means trùng nhau); (b) quá ít dữ liệu so với số tham số (nhiều state, full covariance) → một state chỉ ôm vài frame, phương sai co về `min_covar` (nghiệm suy biến); (c) đặc trưng chưa chuẩn hoá hoặc có silence dài làm một state "nuốt" khoảng lặng; (d) dữ liệu train lẫn file lỗi/nhãn sai; (e) topology bị phá nếu cho phép cập nhật tự do mọi transition.

**4. Vì sao diagonal covariance được ưu tiên khi dữ liệu nhỏ và số chiều cao?** Full covariance cần $D(D+1)/2=780$ tham số/state với D = 39, diag chỉ cần 39. Với vài trăm frame/state, ước lượng full dễ suy biến (không khả nghịch) và overfit (thí nghiệm E3); các hệ số MFCC (sau DCT + CMVN) vốn ít tương quan nên mất mát do bỏ hiệp phương sai là nhỏ. Muốn mô hình phức tạp hơn thì tăng số Gaussian (GMM diag) thay vì dùng full.

**5. Vì sao delta features hữu ích dù HMM đã mô hình hoá chuỗi theo thời gian?** HMM giả thiết output independence: biết state thì các frame độc lập, nên trong một state model không biết phổ đang *đi lên hay đi xuống*. Δ và Δ² đưa thông tin quỹ đạo (vận tốc, gia tốc của phổ qua ~9 frame) vào từng vector, bù lại giả thiết đó và phân biệt tốt các chuyển tiếp phụ âm–nguyên âm, nguyên âm đôi, âm cuối (thí nghiệm E2).

**6. Whole-word HMM không cần lexicon khi nào; khi nào lexicon bắt buộc?** Khi từ vựng nhỏ, cố định và có đủ mẫu thu cho **từng** từ (lệnh điều khiển, chữ số) – mỗi từ có model riêng nên không cần biết cách phát âm. Lexicon bắt buộc khi dùng subword/phone HMM, từ vựng lớn hoặc mở, cần thêm từ mới mà không thu âm lại, hay khi nhận dạng tiếng nói liên tục (decoder phải biết nối phone thành từ).

**7. Nếu test utterance dài hơn do nói chậm, self-loop giúp gì?** Self-loop $a_{ii}$ cho phép ở lại một state nhiều frame liên tiếp, nên cùng một model khớp được utterance dài/ngắn khác nhau: chỉ cần kéo dài thời gian ở các state (chi phí là thêm các thừa số $a_{ii}$ trong likelihood). Đây là "time warping ngầm", tương tự đường ngang/dọc trong DTW. Duration ở mỗi state theo phân bố hình học với kỳ vọng $1/(1-a_{ii})$.

**8. Với cùng một utterance có cần chia log-likelihood cho T để chọn model tốt nhất không?** Không. Mọi model chấm cùng một X với cùng T nên chia cho T không đổi thứ tự argmax. Chia cho T chỉ cần khi so sánh score giữa **các utterance khác độ dài** (đặt ngưỡng reject, vẽ phân bố score) hoặc khi cộng với LM score như mục G.

**9. So sánh warping path của DTW với state path của Viterbi.** Cả hai là đường đơn điệu tìm bằng quy hoạch động. Warping path căn từng frame test với từng frame của **một template cụ thể** (lưới $T_x\times T_y$, chi phí khoảng cách). Viterbi path căn từng frame với **state** của một mô hình thống kê (lưới $T\times N$, điểm là log-xác suất), trong đó mỗi state đại diện cho phân bố của nhiều frame từ mọi utterance train. Do đó Viterbi path gọn hơn (N nhỏ), có phạt duration qua $a_{ij}$ và tổng quát hoá tốt hơn; warping path cụ thể hơn nhưng phụ thuộc chất lượng template.

**10. Từ ngoài từ vựng – vì sao argmax vẫn chọn một lớp, đề xuất reject.** Quy tắc $\arg\max_w P(X\mid\lambda_w)$ chỉ so sánh *tương đối* giữa các model đã có, nên luôn trả về một lớp kể cả khi mọi likelihood đều rất thấp. Đề xuất: từ chối nếu log-likelihood/frame của top-1 < θ (θ chọn tại EER, mục H), hoặc margin/frame giữa top-1 và top-2 < δ; tốt hơn nữa là thêm filler/garbage model và từ chối khi $\log P(X\mid\lambda_{best})-\log P(X\mid\lambda_{filler})$ nhỏ hơn ngưỡng.